In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from firedrake import *
import matplotlib.pyplot as plt
import numpy as np
from math import ceil

from firedrake.petsc import PETSc
import petsc4py

from time import perf_counter

---
---
# Exercise 1
## Solve unsteady Navier-Stokes problem with semi-implicit treatment of the advection term.

\begin{equation*}
\begin{cases}
\frac{\partial\boldsymbol{u}}{\partial t} + (\boldsymbol{u}\cdot\nabla)\boldsymbol{u} - \frac{1}{\rm Re}\Delta \boldsymbol{u} + \nabla  p  = \boldsymbol{f} & {\rm in} \ \Omega=(0,1)^2
\quad\forall t\in(0,1), \\
{\rm div}\,\boldsymbol{u} = 0 & {\rm in} \ \Omega
\quad\forall t\in(0,1), \\
\boldsymbol{u} = \boldsymbol{u}_\text{ex} & {\rm on} \ \partial\Omega
\quad\forall t\in(0,1), \\
\boldsymbol{u} = \boldsymbol{0} & {\rm in} \ \Omega,
\quad{\rm for} \ t=0.
\end{cases}
\end{equation*}

In [ ]:
def ex1_assemble_Euler(V, Q, Re, n, dt, t, u_old, u_ex, p_ex, f, create_pc_form=None, get_nsp=False):
    # Solve monolithic problem.
    W = MixedFunctionSpace([V, Q])

    # Trial and test functions.
    u, p = TrialFunctions(W)
    v, q = TestFunctions(W)

    # Variational form and functional
    a = Constant(1.0/dt) * inner(u, v) * dx \
        - div(v) * p * dx \
        + div(u) * q * dx
    L = Constant(1.0/dt) * inner(u_old, v) * dx \
        - inner(dot(grad(u_old), u_old), v) * dx \
        - Constant(1.0/Re) * inner(grad(u_old), grad(v)) * dx \
        + inner(f, v) * dx

    # Dirichlet BC's
    bcs = ( DirichletBC(W.sub(0), u_ex, (1,2,3,4)) )

    # Define problem, possibly preconditioned.
    wh = Function(W)
    if (create_pc_form):
      pc_form = create_pc_form(u, p, v, q, dt, Re, u_old, dx)
      vpb = LinearVariationalProblem(a, L, wh, bcs, aP=pc_form)
    else:
      vpb = LinearVariationalProblem(a, L, wh, bcs)

    # Create null space, if required.
    if (get_nsp):
      nsp = MixedVectorSpaceBasis(
          W, [W.sub(0), VectorSpaceBasis(constant=True,
                                     comm=COMM_WORLD)] # to suppress warnings
      )
      return wh, vpb, nsp

    else:
      return wh, vpb

In [ ]:
# Parameters
n = 20
t0 = 0
T = 0.12
dt = 0.04

# Problem setup
mesh = UnitSquareMesh(n, n)
x = SpatialCoordinate(mesh)
V = VectorFunctionSpace(mesh, 'P', 2)
Q = FunctionSpace(mesh, 'P', 1)

# Data
Re = 10
sine_2_t = Constant(sin(2*t0))
cosine_2_t = Constant(cos(2*t0))
u_ex = as_vector((
    -cos(x[0]) * sin(x[1]) * sine_2_t,
    sin(x[0]) * cos(x[1]) * sine_2_t))
p_ex = -0.25*(cos(2*x[0])+cos(2*x[1]))*sine_2_t*sine_2_t
f = as_vector((
    -2 * cos(x[0]) * sin(x[1]) * ( cosine_2_t + sine_2_t/Re ),
    2 * sin(x[0]) * cos(x[1]) * ( cosine_2_t + sine_2_t/Re )))

# Solver parameters
params = {'ksp_type': 'gmres', 'pc_type': "fieldsplit",
          "pc_fieldsplit_type": "additive",
          "fieldsplit_0_ksp_type": "preonly", "fieldsplit_0_pc_type": "lu", # block P0 is "inverted" by LU factorization
          "fieldsplit_1_ksp_type": "preonly", "fieldsplit_1_pc_type": "lu", # block P1 is "inverted" by LU factorization
          'ksp_rtol': 1.e-8, 'ksp_max_it': 10000}

# Create preconditioner form
# (we can use the same for additive and multiplicative...)
def create_pc_form_Euler(u, p, v, q, dt, Re, u_old, dx):
  return Constant(1.0/dt) * inner(u, v) * dx \
          + q * div(u) * dx \
          + Constant(Re) * p*q *dx + Constant(dt) * inner(grad(p), grad(q)) * dx

In [ ]:
# Initial condition.
u0 = Function(V)
u0.interpolate(Constant((0., 0.)))
print('t =', t0, ' :  ||u||_{H^1} =', norm(u0,'H1'))

# Define problem and solver
wh, vpb, nsp = ex1_assemble_Euler(V, Q, Re, n, dt, t0, u0, u_ex, p_ex, f, create_pc_form=create_pc_form_Euler, get_nsp=True)

In [ ]:
def solve_time_dependent(u_ex, p_ex, f, sine_2_t, cosine_2_t, t0, T,  wh, vpb, nsp, params, u0, u_oldold=None):

    # Initial conditions
    sine_2_t.assign(sin(2*t0))
    cosine_2_t.assign(cos(2*t0))
    u0.interpolate(Constant((0., 0.)))
    err_u_LinfL2 = 0
    err_u_LinfH1 = 0
    err_p_LinfL2 = 0
    print('t =', t0, ' :  ||u||_{H^1} =', norm(u0,'H1'))

    if (u_oldold): # two-step method
      # First time step: still initialization.
      t = t0+dt
      # ... do something ...
      print('t =', t0, ' :  ||u||_{H^1} =', norm(u0,'H1'))
      t_start = t0+2*dt   # Start from SECOND time step.
    else:
      t_start = t0+dt

    # Time loop
    for t in np.arange(t_start, T+0.1*dt, dt):  # T+0.1*dt to include also T: range/arange exclude the upper bound of the range
        print('t =', t)
        sine_2_t.assign(sin(2*t))
        cosine_2_t.assign(cos(2*t))

        # Advance in time
        elapsed_start = perf_counter()
        solver =  LinearVariationalSolver(vpb, nullspace=nsp, solver_parameters=params)
        solver.solve()
        print('SOLVER   -   elapsed time =', perf_counter() - elapsed_start, 's    -    # iter =', solver.snes.ksp.getIterationNumber())
        uh, ph = wh.subfunctions
        print('SOLUTION -   ||u||_{H^1} =', norm(uh,'H1'), '  (||u_ex||_{H^1} =', norm(Function(V).interpolate(u_ex), 'H1'), ')')

        # fig, ax = plt.subplots()
        # col = tripcolor(ph, axes=ax)
        # plt.colorbar(col)
        # plt.title('pressure - t='+str(t))
        # fig, ax = plt.subplots()
        # # triplot(mesh, axes=ax)
        # col = quiver(uh, axes=ax)
        # plt.colorbar(col)
        # plt.title('velocity - t='+str(t))

        # Compute error
        err_u_LinfL2 = max(err_u_LinfL2, errornorm(u_ex, uh, 'L2'))
        err_u_LinfH1 = max(err_u_LinfH1, errornorm(u_ex, uh, 'H1'))
        # err_p_LinfL2 = max(err_p_LinfL2, errornorm(p_ex, ph, 'L2'))
        # For p, to correctly observe convergence we know that a correction is required in the fully Dirichlet case: see Lab 3.
        mean_ph = assemble(ph*dx)
        mean_p_ex = assemble(p_ex*dx)
        err_p_LinfL2 = max(err_p_LinfL2, sqrt(assemble((ph-mean_ph - (p_ex-mean_p_ex))*(ph-mean_ph - (p_ex-mean_p_ex))*dx)))

        # Update the old solution

        if (u_oldold): # two-step method: mind the order of assignment!
          u_oldold.assign(u0)

        u0.assign(uh)

    print('Errors:', err_u_LinfL2, err_u_LinfH1, err_p_LinfL2)
    return err_u_LinfL2, err_u_LinfH1, err_p_LinfL2

In [ ]:
solve_time_dependent(u_ex, p_ex, f, sine_2_t, cosine_2_t, t0, T, wh, vpb, nsp, params, u0)

---
---
# Point 4 - Convergence analysis - Euler

In [ ]:
# Solve time-dependent problem.
t0 = 0
T = 0.12
dt_vec = [0.04 * 2**ii for ii in range(0, -5, -1)]
err_u_LinfL2 = [0] * len(dt_vec) # NB the command 0*dt_vec would return a scalar
err_u_LinfH1 = [0] * len(dt_vec)
err_p_LinfL2 = [0] * len(dt_vec)

# Convergence loop.
for ii in range(0, len(dt_vec)):
    dt = dt_vec[ii]
    print('##### dt =', dt, '#####', flush=True)
    u0.interpolate(Constant((0., 0.)))
    wh, vpb, nsp = ex1_assemble_Euler(V, Q, Re, n, dt, t0, u0, u_ex, p_ex, f, create_pc_form=create_pc_form_Euler, get_nsp=True)

    err_u_LinfL2[ii], err_u_LinfH1[ii], err_p_LinfL2[ii] = solve_time_dependent(u_ex, p_ex, f, sine_2_t, cosine_2_t, t0, T, wh, vpb, nsp, params, u0)

In [ ]:
print('Errors:', err_u_LinfL2, err_u_LinfH1, err_p_LinfL2)
# print('# iterations:', max_it_schur)
plt.figure()
plt.loglog(dt_vec, err_u_LinfL2, marker='o', label='u Linf-L2')
plt.loglog(dt_vec, err_u_LinfH1, marker='o', label='u Linf-H1')
plt.loglog(dt_vec, err_p_LinfL2, marker='+', label='p Linf-L2')
plt.loglog(dt_vec, dt_vec, linestyle='--', label='dt')

plt.legend()

---
---
# Point 4 - Convergence analysis - BDF2

In [ ]:
...